In [ ]:
!nvidia-smi

In [ ]:
import sys
import torch
import types
sys.path.append("..")
from typing import List, Optional
from diffsynth.models.wan_video_dit_causal_animate import CausalWanModel as causal_wan_model
from diffsynth.models.wan_video_vae import VideoVAE_ as vae_model
from diffsynth.models.wan_video_text_encoder import WanTextEncoder as text_encoder
from diffsynth.models.wan_video_image_encoder import WanImageEncoder as image_encoder
from diffsynth.models.wan_video_text_encoder import HuggingfaceTokenizer
from diffsynth.scheduler import SchedulerInterface, FlowMatchScheduler
from diffsynth.core.loader.file import load_state_dict

model_paths = [["/primus_xpfs_workspace_T04/zyx/models/Wan2.2-Animate-14B/diffusion_pytorch_model-00001-of-00004.safetensors", "/primus_xpfs_workspace_T04/zyx/models/Wan2.2-Animate-14B/diffusion_pytorch_model-00002-of-00004.safetensors", "/primus_xpfs_workspace_T04/zyx/models/Wan2.2-Animate-14B/diffusion_pytorch_model-00003-of-00004.safetensors", "/primus_xpfs_workspace_T04/zyx/models/Wan2.2-Animate-14B/diffusion_pytorch_model-00004-of-00004.safetensors"], "/primus_xpfs_workspace_T04/zyx/models/Wan2.2-Animate-14B/models_t5_umt5-xxl-enc-bf16.pth", "/primus_xpfs_workspace_T04/zyx/models/Wan2.2-Animate-14B/Wan2.1_VAE.pth", "/primus_xpfs_workspace_T04/zyx/models/Wan2.2-Animate-14B/models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth"]
tokenizer_path = "/primus_xpfs_workspace_T04/zyx/models/Wan2.2-Animate-14B/google/umt5-xxl"
default_dtype = torch.bfloat16

def init_dit_model():
    kwargs = {
        "dim": 5120,
        "eps": 1e-06,
        "ffn_dim": 13824,
        "freq_dim": 256,
        "in_dim": 36,
        "num_heads": 40,
        "num_layers": 40,
        "out_dim": 16,
        "patch_size": [1, 2, 2],
        "text_dim": 4096,
        "has_image_input": True
    }
    state_dict = load_state_dict(file_path=model_paths[0], torch_dtype=torch.bfloat16, device="cpu")
    model = causal_wan_model(**kwargs)
    model.load_state_dict(state_dict, strict=True)
    return model

def init_vae_model():
    vae_state_dict = load_state_dict(model_paths[2], torch_dtype=torch.bfloat16, device="cpu")
    vae_model_instance = vae_model()
    vae_model_instance.load_state_dict(vae_state_dict, strict=True)
    return vae_model_instance

def init_text_encoder_model():
    text_encoder_state_dict = load_state_dict(model_paths[1], torch_dtype=torch.bfloat16, device="cpu")
    text_encoder_model_instance = text_encoder()
    text_encoder_model_instance.load_state_dict(text_encoder_state_dict, strict=True)
    return text_encoder_model_instance

def init_clip_model():
    clip_state_dict = load_state_dict(model_paths[3], torch_dtype=torch.bfloat16, device="cpu")
    fixed_state = {}
    for k, v in clip_state_dict.items():
        new_k = k if k.startswith("model.") else "model." + k
        fixed_state[new_k] = v
    filtered = {k: v for k, v in fixed_state.items() if k.startswith("model.visual.") or k == "model.log_scale"}
    clip_model_instance = image_encoder()
    clip_model_instance.load_state_dict(filtered, strict=True)
    return clip_model_instance

def init_tokenizer():
    tokenizer = HuggingfaceTokenizer(name=tokenizer_path, seq_len=512, clean='whitespace')
    return tokenizer


class WanTextEncoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.text_encoder = init_text_encoder_model().eval().requires_grad_(False).to(dtype=default_dtype)
        self.tokenizer = init_tokenizer()
        
    @property
    def device(self):
        # Assume we are always on GPU
        return torch.cuda.current_device()
        
    def forward_text(self, text_prompts: List[str]) -> dict:
        ids, mask = self.tokenizer(
            text_prompts, return_mask=True, add_special_tokens=True)
        ids = ids.to(self.device)
        mask = mask.to(self.device)
        seq_lens = mask.gt(0).sum(dim=1).long()
        context = self.text_encoder(ids, mask)
        for u, v in zip(context, seq_lens):
            u[v:] = 0.0  # set padding to 0.0
        return context
    
    
class WanVAEWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.vae = init_vae_model().eval().requires_grad_(False).to(dtype=default_dtype)
        mean = [
            -0.7571, -0.7089, -0.9113, 0.1075, -0.1745, 0.9653, -0.1517, 1.5508,
            0.4134, -0.0715, 0.5517, -0.3632, -0.1922, -0.9497, 0.2503, -0.2921
        ]
        std = [
            2.8184, 1.4541, 2.3275, 2.6558, 1.2196, 1.7708, 2.6052, 2.0743,
            3.2687, 2.1526, 2.8652, 1.5579, 1.6382, 1.1253, 2.8251, 1.9160
        ]
        self.mean = torch.tensor(mean, dtype=torch.float32)
        self.std = torch.tensor(std, dtype=torch.float32)

    def encode_to_latent(self, pixel: torch.Tensor) -> torch.Tensor:
        # pixel: (B, C, T, H, W)
        device, dtype = pixel.device, pixel.dtype
        scale = [self.mean.to(device=device, dtype=dtype),
                 1.0 / self.std.to(device=device, dtype=dtype)]

        output = [
            self.model.encode(u.unsqueeze(0), scale).float().squeeze(0)
            for u in pixel
        ]
        output = torch.stack(output, dim=0)
        output = output.permute(0, 2, 1, 3, 4)
        # output: (B, T, C, H, W)
        return output

    def decode_to_pixel(self, latent: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        # latent: (B, T, C, H, W)
        zs = latent.permute(0, 2, 1, 3, 4)
        if use_cache:
            assert latent.shape[0] == 1, "Batch size must be 1 when using cache"

        device, dtype = latent.device, latent.dtype
        scale = [self.mean.to(device=device, dtype=dtype),
                 1.0 / self.std.to(device=device, dtype=dtype)]

        if use_cache:
            decode_function = self.model.cached_decode
        else:
            decode_function = self.model.decode

        output = []
        for u in zs:
            output.append(decode_function(u.unsqueeze(0), scale).float().clamp_(-1, 1).squeeze(0))
        output = torch.stack(output, dim=0)
        output = output.permute(0, 2, 1, 3, 4)
        # output: (B, C, T, H, W)
        return output


class WanImageEncoderWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.image_encoder = init_clip_model().eval().requires_grad_(False).to(dtype=default_dtype)
        
    @property
    def device(self):
        # Assume we are always on GPU
        return torch.cuda.current_device()
        
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.image_encoder.encode_image(images)
    

class WanDiffusionWrapper(torch.nn.Module):
    def __init__(
            self,
            timestep_shift=3.0,
            is_causal=False,
            local_attn_size=-1,
            num_frame_per_block=3
    ):
        super().__init__()

        self.model = init_dit_model()
        # For non-causal diffusion, all frames share the same timestep
        self.uniform_timestep = not is_causal

        self.scheduler = FlowMatchScheduler(
            shift=timestep_shift, sigma_min=0.0, extra_one_step=True
        )
        self.scheduler.set_timesteps(1000, training=True)

        self.seq_len =  32760 # [1, 21, 16, 60, 104]
        self.num_frame_per_block = num_frame_per_block
        self.local_attn_size = local_attn_size
        self.post_init()

    def enable_gradient_checkpointing(self) -> None:
        self.model.enable_gradient_checkpointing()

    def _convert_flow_pred_to_x0(self, flow_pred: torch.Tensor, xt: torch.Tensor, timestep: torch.Tensor) -> torch.Tensor:
        """
            Convert flow matching's prediction to x0 prediction.
            flow_pred: the prediction with shape [B, C, H, W]
            xt: the input noisy data with shape [B, C, H, W]
            timestep: the timestep with shape [B]

            pred = noise - x0
            x_t = (1-sigma_t) * x0 + sigma_t * noise
            we have x0 = x_t - sigma_t * pred
            see derivations https://chatgpt.com/share/67bf8589-3d04-8008-bc6e-4cf1a24e2d0e
        """
        # use higher precision for calculations
        original_dtype = flow_pred.dtype
        flow_pred, xt, sigmas, timesteps = map(
            lambda x: x.double().to(flow_pred.device), [flow_pred, xt,
                                                        self.scheduler.sigmas,
                                                        self.scheduler.timesteps]
        )

        timestep_id = torch.argmin(
            (timesteps.unsqueeze(0) - timestep.unsqueeze(1)).abs(), dim=1)
        sigma_t = sigmas[timestep_id].reshape(-1, 1, 1, 1)
        x0_pred = xt - sigma_t * flow_pred
        return x0_pred.to(original_dtype)

    @staticmethod
    def _convert_x0_to_flow_pred(scheduler, x0_pred: torch.Tensor, xt: torch.Tensor, timestep: torch.Tensor) -> torch.Tensor:
        """
        Convert x0 prediction to flow matching's prediction.
        x0_pred: the x0 prediction with shape [B, C, H, W]
        xt: the input noisy data with shape [B, C, H, W]
        timestep: the timestep with shape [B]

        pred = (x_t - x_0) / sigma_t
        """
        # use higher precision for calculations
        original_dtype = x0_pred.dtype
        x0_pred, xt, sigmas, timesteps = map(
            lambda x: x.double().to(x0_pred.device), [x0_pred, xt,
                                                      scheduler.sigmas,
                                                      scheduler.timesteps]
        )
        timestep_id = torch.argmin(
            (timesteps.unsqueeze(0) - timestep.unsqueeze(1)).abs(), dim=1)
        sigma_t = sigmas[timestep_id].reshape(-1, 1, 1, 1)
        flow_pred = (xt - x0_pred) / sigma_t
        return flow_pred.to(original_dtype)

    def forward(
        self,
        noisy_image_or_video: torch.Tensor, conditional_dict: dict,
        timestep: torch.Tensor, kv_cache: Optional[List[dict]] = None,
        crossattn_cache: Optional[List[dict]] = None,
        current_start: Optional[int] = None,
        classify_mode: Optional[bool] = False,
        concat_time_embeddings: Optional[bool] = False,
        clean_x: Optional[torch.Tensor] = None,
        aug_t: Optional[torch.Tensor] = None,
        cache_start: Optional[int] = None #疑似没用
    ) -> torch.Tensor:
        # [B, F] -> [B]
        if self.uniform_timestep:
            input_timestep = timestep[:, 0]
        else:
            input_timestep = timestep

        if kv_cache is not None:
            flow_pred = self.model(
                noisy_image_or_video.permute(0, 2, 1, 3, 4),
                t=input_timestep, **conditional_dict,
                kv_cache=kv_cache,
                crossattn_cache=crossattn_cache,
                current_start=current_start
            )
        else:
            flow_pred = self.model(
                noisy_image_or_video.permute(0, 2, 1, 3, 4),
                t=input_timestep, **conditional_dict
            )

        flow_pred = torch.stack(flow_pred).permute(0, 2, 1, 3, 4)
        pred_x0 = self._convert_flow_pred_to_x0(
            flow_pred=flow_pred.flatten(0, 1),
            xt=noisy_image_or_video.flatten(0, 1),
            timestep=timestep.flatten(0, 1)
        ).unflatten(0, flow_pred.shape[:2])

        return flow_pred, pred_x0

    def get_scheduler(self) -> SchedulerInterface:
        """
        Update the current scheduler with the interface's static method
        """
        scheduler = self.scheduler
        scheduler.convert_x0_to_noise = types.MethodType(
            SchedulerInterface.convert_x0_to_noise, scheduler)
        scheduler.convert_noise_to_x0 = types.MethodType(
            SchedulerInterface.convert_noise_to_x0, scheduler)
        scheduler.convert_velocity_to_x0 = types.MethodType(
            SchedulerInterface.convert_velocity_to_x0, scheduler)
        self.scheduler = scheduler
        return scheduler

    def post_init(self):
        """
        A few custom initialization steps that should be called after the object is created.
        Currently, the only one we have is to bind a few methods to scheduler.
        We can gradually add more methods here if needed.
        """
        self.get_scheduler()
        self.model.num_frame_per_block = self.num_frame_per_block

In [ ]:
# WanTextEncoder()
# WanVAEWrapper()
# WanImageEncoderWrapper()
# WanAnimateAdapterWrapper()
model = WanDiffusionWrapper().to(device=torch.device("cuda:7"), dtype=torch.bfloat16)

In [ ]:
for param_name, param in model.named_parameters():
    print(param_name, param.device)

In [ ]:
#### lora utils from diffsys ####
from diffsynth.core import load_state_dict, AutoWrappedLinear

def add_lora_to_model(
    model,
    lora_rank=128,
    alpha=1, # default 1 (rank=alpha)
    pretrained_lora_path=None,
    lora_target_modules="q,k,v,o,ffn.0,ffn.2",
):

    from peft import LoraConfig, inject_adapter_in_model
    if alpha == 1:
        lora_alpha = lora_rank
    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_alpha,
        init_lora_weights=True,
        target_modules=lora_target_modules.split(","),
    )
    model = inject_adapter_in_model(lora_config, model)
    
    if pretrained_lora_path is not None:
        print(f"Loading pretrained LoRA from {pretrained_lora_path}...")
        lora = load_state_dict(pretrained_lora_path)
        def fix_lora_state_dict_keys(lora_state_dict):
            fixed = {}
            for k, v in lora_state_dict.items():
                if not k.startswith("model."):
                    fixed["model." + k] = v
                else:
                    fixed[k] = v
            return fixed
        lora = fix_lora_state_dict_keys(lora)
        missing_keys, unexpected_keys = model.load_state_dict(lora, strict=False)
        print(f"Loaded pretrained LoRA. Missing keys: {missing_keys}, Unexpected keys: {unexpected_keys}")
    
    return model

def clear_lora(model, verbose=1):
    cleared_num = 0
    for name, module in model.named_modules():
        if isinstance(module, AutoWrappedLinear):
            if hasattr(module, "lora_A_weights"):
                if len(module.lora_A_weights) > 0:
                    cleared_num += 1
                module.lora_A_weights.clear()
            if hasattr(module, "lora_B_weights"):
                module.lora_B_weights.clear()
    if verbose >= 1:
        print(f"{cleared_num} LoRA layers are cleared.")
    return model
        
def get_name_dict(lora_state_dict):
    lora_name_dict = {}
    for key in lora_state_dict:
        if ".lora_up." in key:
            lora_A_key = "lora_down"
            lora_B_key = "lora_up"
        else:
            lora_A_key = "lora_A"
            lora_B_key = "lora_B"
        if lora_B_key not in key:
            continue
        keys = key.split(".")
        if len(keys) > keys.index(lora_B_key) + 2:
            keys.pop(keys.index(lora_B_key) + 1)
        keys.pop(keys.index(lora_B_key))
        if keys[0] == "diffusion_model":
            keys.pop(0)
        keys.pop(-1)
        target_name = ".".join(keys)
        lora_name_dict[target_name] = (key, key.replace(lora_B_key, lora_A_key))
    return lora_name_dict

def convert_state_dict(state_dict, suffix=".weight"):
    name_dict = get_name_dict(state_dict)
    state_dict_ = {}
    for name in name_dict:
        weight_up = state_dict[name_dict[name][0]]
        weight_down = state_dict[name_dict[name][1]]
        state_dict_[name + f".lora_B{suffix}"] = weight_up
        state_dict_[name + f".lora_A{suffix}"] = weight_down
    return state_dict_
    
def fuse_lora_to_base_model(model: torch.nn.Module, state_dict, alpha=1.0):
    updated_num = 0
    state_dict = convert_state_dict(state_dict)
    lora_layer_names = set([i.replace(".lora_B.weight", "") for i in state_dict if i.endswith(".lora_B.weight")])
    for name, module in model.named_modules():
        if name in lora_layer_names:
            param = next(module.parameters())
            weight_up = state_dict[name + ".lora_B.weight"].to(device=param.device, dtype=param.dtype)
            weight_down = state_dict[name + ".lora_A.weight"].to(device=param.device, dtype=param.dtype)
            if len(weight_up.shape) == 4:
                weight_up = weight_up.squeeze(3).squeeze(2)
                weight_down = weight_down.squeeze(3).squeeze(2)
                weight_lora = alpha * torch.mm(weight_up, weight_down).unsqueeze(2).unsqueeze(3)
            else:
                weight_lora = alpha * torch.mm(weight_up, weight_down)
            state_dict_base = module.state_dict()
            state_dict_base["weight"] = state_dict_base["weight"].to(device=param.device, dtype=param.dtype) + weight_lora
            module.load_state_dict(state_dict_base)
            updated_num += 1
    print(f"{updated_num} tensors are fused by LoRA. Fused LoRA layers cannot be cleared by `clear_lora()`.")

In [ ]:
model_with_lora = add_lora_to_model(model, lora_rank=128, pretrained_lora_path=None)
for name, module in model_with_lora.named_modules():
    if ".lora_A." in name or ".lora_B." in name:
        print(name, module.weight)

In [ ]:
def clear_lora(model, verbose=1):
    cleared_num = 0
    for name, param in model.named_parameters():
        if ".lora_A." in name or ".lora_B." in name:
            param.data.zero_()
            cleared_num += 1
    if verbose >= 1:
        print(f"{cleared_num} LoRA parameters are zeroed.")
    return model

model_with_lora = clear_lora(model_with_lora, verbose=1)
for name, module in model_with_lora.named_modules():
    if ".lora_A." in name or ".lora_B." in name:
        print(name, module.weight)

In [ ]:
for name, module in model_with_lora.named_modules():
    if ".lora_A." in name or ".lora_B." in name:
        print(name, module.weight)

In [ ]:
pretrained_lora_path="/primus_xpfs_workspace_T04/zyx/projects/project2/DiffSynth-Studio/causal_models/step-7000.safetensors"
lora = load_state_dict(pretrained_lora_path)
for name, param in lora.items():
    print(name, param)

In [ ]:
model_with_lora = add_lora_to_model(model, lora_rank=128, pretrained_lora_path="/primus_xpfs_workspace_T04/zyx/projects/project2/DiffSynth-Studio/causal_models/step-7000.safetensors")
for name, module in model_with_lora.named_modules():
    if ".lora_A." in name or ".lora_B." in name:
        print(name, module.weight)